# This jupyter notebook file will create a .txt file in ITOL format to color genome branches of an ITOL tree based on specific information you provide

# Running all code in this file will produce 1 text file, which is capable of creating 1 ring of colors around an ITOL tree. To make multiple rings with different annotations/colors, run this code multiple times with different inputs.

# To start you need:
## - 1st .csv file with two columns:
### --(1) all genome names for all branches on your tree (must be identical to ITOL node IDs)
###    --(2) information you want to annotate per genome (ex. allele # present in each genome, presence/absence information for each genome, etc.)
## -2nd .csv file with two columns:
###    --(1) all possible variations (including a blank, if present) for the information you want to annotate per genome (ex. if you are annotating allele # have 10 alleles, this column would have numbers 1-10)
###    --(2) a hex color code (including '#' character) to assign to each information value (ex. next to allele 1 from column 1, you put '#00ff00' in column 2)

# Import necessary packages

In [3]:
# import necessary packages
import os
import pandas as pd
import numpy as np

# Define your working directory

In [4]:
# Define working directory (the folder that contains your input files)
folder = '/home/xinruoz/Document/Hiller/surface_chaperones/'
#example: folder = 'C:/filepath_for_directory_with_your_File1_and_File2/'

# Get information from your 1st csv file

In [ ]:
# get my cvs for ITOL tree

# get all the species and their name

# read in the allele info

# output the needed csv file for ITOL tree


In [ ]:
# open first .csv file
file1 = './output/Spn_Tree_TprA_25June25_ex.csv'
#example: file1 = 'File1.csv'
#example (if file is in a subdirectory of your working directory, here named 'Inputs'): file1 = './Inputs/Spn_Tree_TprA_25June25.csv'

# make first .csv file into a pandas dataframe (df)
d1 = pd.read_csv(folder+file1)
# visualize your df
d1

,Genomes ID,Downstream tprA
0,1235_3063_00,sce-bac; Other's
1,13176_2682,contig break
2,15432_46_2,sce-IM
3,15434_35F_2,contig break
4,15435_11B_2,sce-IM
...,...,...
7541,49211_SC_2440,NaN
7542,49212_SC_2439,NaN
7543,49213_SC_2441,sce-IM
7544,6630_Kor_136,contig break


# Check output of the next block!!

In [72]:
# This block will ensure there are no duplicate genomes in your 1st csv file.
# If no duplicates, output will be "all gud"
# If there are duplicates, output will show you what the duplicates are.


# Create a list of headers from your 1st csv file
d1_headers = d1.columns.to_list()

# Create a list of all your genomes
genomes = d1[d1_headers[0]].to_list()
# Convert list to set to remove duplicate names
unique_genomes_set = set(genomes)

# check if the length of the list of all genomes is identical to list of unique genome names
# if true, output = "all gud"
if len(genomes) == len (unique_genomes_set):
    print('all gud')
# if false, output = "ruh roh dups exist", then a df which shows you the duplicated genome names
else:
    print('ruh roh dups exist')
    duplicate_rows = d1[d1['Genomes ID'].duplicated(keep=False)]
    print("\nRows with duplicate values in 'Genomes ID':")
    print(duplicate_rows)

all gud


# Run block below only for Spn. This will trigger an error for Sm.
#### If working with Sm, just comment out the block so it does not run.
#### If working with Spn and you skip this, you will get an error when you try to import to ITOL, because the Spn poppunk tree we have is missing this one genome from the GoldenSet '35860_Spain9V_3'. The Spn poppunk tree has 7547 branches.

In [73]:
# remove genome 35860_Spain9V_3 from dataset
d1 = d1[d1[d1_headers[0]] != '35860_Spain9V_3']

# Get information from your 2nd csv file

In [74]:
# open second .csv file
file2 = './Inputs/b_downstream_colorcode.csv'
#example: file1 = 'File1.csv'
#example (if file is in a subdirectory of your working directory, here named 'Inputs'): file1 = './Inputs/b_downstream_colorcode.csv'

# make first .csv file into a pandas dataframe (df)
d2 = pd.read_csv(folder+file2)
# visualize your df
d2

,Category,HEX Color
0,sce-bac; Other's,#70A17F
1,Tn916-like; sce-IM,#66C2A4
2,Tn916-like; sce-bac; sce-IM,#A8D5BA
3,cbpG-like; lan-like,#94DCF8
4,Tn916-like; lan-like,#00CBC4
5,sce-IM,#006D2C
6,sce-bac; sce-IM,#23A745
7,lan-like;sce-IM,#9AAA5B
8,cbpG-like; sce-IM,#B6C376
9,cbpG-like;sce-IM,#B6C376


In [75]:
# Convert your second .csv file into a dictionary to be used to assign HEX color codes

# Create a list of headers from your 2nd csv file
d2_headers = d2.columns.to_list()

# Make d2 into a dictionary with column 1 values as keys and column 2 values as values for those keys
d2_dict = d2.set_index(d2_headers[0])[d2_headers[1]].to_dict()
# Visualize dictionary to ensure color codes from 2nd column match up with values in 1st column
d2_dict

{"sce-bac; Other's": '#70A17F',
 'Tn916-like; sce-IM': '#66C2A4',
 'Tn916-like; sce-bac; sce-IM': '#A8D5BA',
 'cbpG-like; lan-like': '#94DCF8',
 'Tn916-like; lan-like': '#00CBC4',
 'sce-IM': '#006D2C',
 'sce-bac; sce-IM': '#23A745',
 'lan-like;sce-IM': '#9AAA5B',
 'cbpG-like; sce-IM': '#B6C376',
 'cbpG-like;sce-IM': '#B6C376',
 'cbpG-like;sce-bac; sce-IM': '#ADD191',
 'lan-like': '#253494',
 'cbpG-like': '#E78AC3',
 'ABC transporter': '#9F90E7',
 'Tn916-like': '#CC4C4C',
 "Other's": '#FFE66D',
 'contig break': '#FDAE6B'}

# Create new df with information from 1st and 2nd .csv files

In [76]:
# Create new df from 1st csv
df_final = d1.copy()

# Make a list of headers for later use
df_final_headers = df_final.columns.tolist()

# Add a new column 'COLOR' in which you assign HEX color codes from your 2nd .csv file, based on values in 2nd column of 1st .csv file 
df_final['COLOR'] = df_final[df_final_headers[1]].map(d2_dict)

# Make list of headers again after adding COLORS column
df_final_headers = df_final.columns.tolist()

df_final

,Genomes ID,Downstream tprA,COLOR
0,1235_3063_00,sce-bac; Other's,#70A17F
1,13176_2682,contig break,#FDAE6B
2,15432_46_2,sce-IM,#006D2C
3,15434_35F_2,contig break,#FDAE6B
4,15435_11B_2,sce-IM,#006D2C
...,...,...,...
7541,49211_SC_2440,NaN,NaN
7542,49212_SC_2439,NaN,NaN
7543,49213_SC_2441,sce-IM,#006D2C
7544,6630_Kor_136,contig break,#FDAE6B


# Check output of the next block!!
#### Any 'NaN values' are blank. These will not be assigned a color in ITOL.
#### This does not cause any errors for ITOL, just make sure all of the genomes you want to be assigned a color will get a color!

### If output it "ur gud"--
#### option 1 --and you DO want to color all genomes, then you are all good!
#### option 2 --and you DO NOT want to color all genomes, this is not good. You are somehow coloring genomes you did not mean to. Double check the values in the 1st column of your 2nd .csv file and ensure you did not assign a HEX code to something you do not want to color.

### If output is "There are NaN values" with a df of genomes--
#### option 1 --and (1) you DO NOT want to color all genomes, and (2) the genomes you do not want to color are listed, then you are all good!
#### option 2 --and (1) you DO want to color all genomes, or (2) at least some of the genomes listed should be colored, this is not good. Diagnose problem in two ways:
#### Fix 1 --Look at the 2nd column in the df below to see which value(s) is not being correctly assigned its HEX code. To fix this, the value(s) in the 1st column of your 2nd .csv file and the value(s) in the 2nd column of your 1st .csv file must be exactly identical. Double check for spaces (sometimes a space is present at the end of a word) and for upper/lower case letters.
#### Fix 2 --If all values in the 2nd column in the df below are also NaN (in addition to all values in 3rd column below), then you have blanks in the 2nd column of your 1st csv file AND you did not assign a HEX color code to a blank value in your 2nd csv file. If you do not want to color those genomes anything, that is okay. If you do wish to color those genomes with something, add another row to your 2nd csv file with no value (blank) in column 1 and a HEX color code in column 2 to assign to these genomes.

In [77]:
# Make sure every genome has a color assigned for downstream categories
df_final_nan = df_final[df_final['COLOR'].isna()]

if not df_final_nan.empty:
    print("There are NaN values")
    print("Rows with NaNs:")
    print(df_final_nan)
else:
    print("ur gud")

There are NaN values
Rows with NaNs:
            Genomes ID Downstream tprA COLOR
87    18785_SMC1205_93             NaN   NaN
381      25724_SMRU773             NaN   NaN
386     25729_SMRU3107             NaN   NaN
389     25732_SMRU1006             NaN   NaN
390      25733_SMRU409             NaN   NaN
...                ...             ...   ...
7534    49197_ECC_3510             NaN   NaN
7535    49202_ECC_3507             NaN   NaN
7538    49207_ECC_1854             NaN   NaN
7541     49211_SC_2440             NaN   NaN
7542     49212_SC_2439             NaN   NaN

[226 rows x 3 columns]


# Edit final df to ITOL format, then export as .txt file

In [78]:
# re-order columns to put COLOR before your information
df_final = df_final[[df_final_headers[0],df_final_headers[2],df_final_headers[1]]]

df_final = df_final.rename(columns={df_final_headers[0]: 'NODE_ID'})
df_final = df_final.rename(columns={df_final_headers[1]: 'LABEL_OR_STYLE'})

# add columns for ITOL format
df_final.insert(1, 'TYPE', 'range')
df_final.insert(4, 'SIZE_FACTOR', '1')
df_final

,NODE_ID,TYPE,COLOR,LABEL_OR_STYLE,SIZE_FACTOR
0,1235_3063_00,range,#70A17F,sce-bac; Other's,1
1,13176_2682,range,#FDAE6B,contig break,1
2,15432_46_2,range,#006D2C,sce-IM,1
3,15434_35F_2,range,#FDAE6B,contig break,1
4,15435_11B_2,range,#006D2C,sce-IM,1
...,...,...,...,...,...
7541,49211_SC_2440,range,NaN,NaN,1
7542,49212_SC_2439,range,NaN,NaN,1
7543,49213_SC_2441,range,#006D2C,sce-IM,1
7544,6630_Kor_136,range,#FDAE6B,contig break,1


# Name your output file below!
#### OPTIONAL: also assign output location. Default save location will be whichever directory this jupyter notebook is located in

In [79]:
# write final dataframe into .txt file with correct format for ITOL annotation 

# IMPORTANT!!!!!!!!!
# By default, this output file will be saved to whichever directory this jupyter notebook is located in
# You can change the save location by adding a filepath before your file name, if desired
# NAME YOUR FILE HERE!!!!!!!!!!!!
txt_output_name = 'example.txt'
# Ex. w/ filepath to save elsewhere: txt_output_name = 'C:/Users/camph/Data/Barbara_ITOL/Tree_annotation_files/example.txt'



# Everything else below is automatically done!
# make .txt file with necessary lines for ITOL annotation
lines = ['TREE_COLORS', 'SEPARATOR TAB', 'DATA']
with open('temp_1.txt', 'w') as f:
    for line in lines:
        f.write(line)
        f.write('\n')
        f.write('\n')

# make .txt file with dataframe
df_final.to_csv('temp_2.txt', sep='\t', index=False, header=False)

# append the content of the two .txt files into one final .txt file
final_txt = ['temp_1.txt', 'temp_2.txt']

#make final txt file
with open(txt_output_name, 'w') as outfile:
    for fname in final_txt:
        with open(fname) as infile:
            for line in infile:
                outfile.write(line)

# remove temp .txt files
os.remove("temp_1.txt")
os.remove("temp_2.txt")